In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
import os

In [2]:
class Our_Dataset(Dataset):
  def __init__(self, root, train=True, transform=None):
    self.root = root
    self.transform = transform
    if train:
      self.labels_frame = pd.read_csv(os.path.join(root, 'labels.csv'))
      self.img_dir = os.path.join(root, 'trainval')
    else:
      self.img_dir = os.path.join(root, 'test')
      self.img_names = sorted(os.listdir(self.img_dir)) #сортируем, чтобы предикты шли в том же порядке, что и в sample_submission
    self.train = train

  def __len__(self):
    if self.train:
      return len(self.labels_frame)
    else:
      return len(self.img_names)

  def __getitem__(self, idx):
    if self.train:
      img_name = os.path.join(self.img_dir, self.labels_frame.iloc[idx, 0])
      label = int(self.labels_frame.iloc[idx, 1])
    else:
      img_name = os.path.join(self.img_dir, self.img_names[idx])
      label = -1 #так как нам сами ответы для теста неизвестны, ставим просто -1, чтобы в любом случае ретернилась пара image, label
    image = Image.open(img_name).convert('RGB')
    if self.transform:
      image = self.transform(image)
    return image, label

In [3]:
def get_our_data(batch_size, transform_train):
    torch.manual_seed(0)
    np.random.seed(0)

    transform_test = transforms.Compose(
        [transforms.ToTensor(),
         # Переводим цвета пикселей в отрезок [-1, 1]
         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    )

    # Загружаем данные
    trainvalset = Our_Dataset(root='/kaggle/input/bhw-1-dl-2025-2026/bhw1', train=True,
                                               transform=transform_train)
    testset = Our_Dataset(root='/kaggle/input/bhw-1-dl-2025-2026/bhw1', train=False,
                                           transform=transform_test)

    # В датасете определено разбиение только на train и test,
    # так что валидацию дополнительно выделяем из обучающей выборки
    train_idx, valid_idx = train_test_split(np.arange(len(trainvalset)), test_size=0.3,
                                            shuffle=True, random_state=0)
    trainset = torch.utils.data.Subset(trainvalset, train_idx)
    valset = torch.utils.data.Subset(trainvalset, valid_idx)

    train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                               shuffle=True, num_workers=2)
    val_loader = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                             shuffle=False, num_workers=2)
    test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                              shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [4]:
transform = transforms.Compose(
        [transforms.RandomHorizontalFlip(),
         transforms.RandomCrop(40, padding=4),
         transforms.TrivialAugmentWide(),
         transforms.ToTensor(),
         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

train_loader, val_loader, test_loader = get_our_data(batch_size=64,
                                                         transform_train=transform)

In [5]:
from torchvision import models
n_classes = 200
net = models.resnet34(weights=None)
net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
net.maxpool = nn.Identity()
net.fc = nn.Linear(net.fc.in_features, n_classes)
net

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [7]:
net = net.to(device)

In [8]:
def test(model, loader):
    loss_log = []
    acc_log = []
    model.eval()

    with torch.no_grad():
      for data, target in loader:
          data = data.to(device)
          target = target.to(device)
          # <your code here>
          logits = model(data)
          loss = F.cross_entropy(logits, target, label_smoothing=0.1)
          loss_log.append(loss.item())

          # <your code here>
          predicts = logits.argmax(dim=1)
          сorr_preds = (predicts == target).float()
          acc = сorr_preds.mean()
          acc_log.append(acc.item())

    return np.mean(loss_log), np.mean(acc_log)

def train_epoch(model, optimizer, train_loader):
    loss_log = []
    acc_log = []
    model.train()

    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)
        # <your code here>
        lam = np.random.beta(0.2, 0.2)
        index = torch.randperm(data.size(0)).to(device)
        mixed_data = lam * data + (1 - lam) * data[index, :]
        
        optimizer.zero_grad()
        logits = model(mixed_data)
        loss = lam * F.cross_entropy(logits, target, label_smoothing=0.1) + (1 - lam) * F.cross_entropy(logits, target[index], label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        loss_log.append(loss.item())

        # <your code here>
        predicts = logits.argmax(dim=1)
        acc = (lam * (predicts == target).float() + (1 - lam) * (predicts == target[index]).float()).mean()
        acc_log.append(acc.item())

    return loss_log, acc_log

def train(model, optimizer, n_epochs, train_loader, val_loader, scheduler=None):
    train_loss_log, train_acc_log, val_loss_log, val_acc_log = [], [], [], []

    for epoch in range(n_epochs):
        train_loss, train_acc = train_epoch(model, optimizer, train_loader)
        val_loss, val_acc = test(model, val_loader)

        train_loss_log.extend(train_loss)
        train_acc_log.extend(train_acc)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)

        print(f"Epoch {epoch}")
        print(f" train loss: {np.mean(train_loss)}, train acc: {np.mean(train_acc)}")
        print(f" val loss: {val_loss}, val acc: {val_acc}\n")

        if scheduler is not None:
            scheduler.step()

    return train_loss_log, train_acc_log, val_loss_log, val_acc_log

In [9]:
def predict(model, loader):

    model.eval()
    predictions = []

    with torch.no_grad():
      for data, _ in loader:
          data = data.to(device)
          # <your code here>
          logits = model(data)

          # <your code here>
          predicts = logits.argmax(dim=1)
          predictions.extend(predicts.cpu().numpy())

    return predictions

In [10]:
optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
train_loss_log, train_acc_log, val_loss_log, val_acc_log = train(net, optimizer, 150, train_loader, val_loader, scheduler)

Epoch 0
 train loss: 5.170727572013953, train acc: 0.018676384127825154
 val loss: 5.047243087784822, val acc: 0.02931769722814499

Epoch 1
 train loss: 4.989085080636703, train acc: 0.036130161065548345
 val loss: 5.111368229140097, val acc: 0.047119314148863244

Epoch 2
 train loss: 4.860074186673766, train acc: 0.055789100032336225
 val loss: 4.772490071335327, val acc: 0.07037357854118734

Epoch 3
 train loss: 4.692710979765033, train acc: 0.0806689983749508
 val loss: 4.480175474813498, val acc: 0.1076981165706476

Epoch 4
 train loss: 4.53185525280466, train acc: 0.10742379094846888
 val loss: 4.409051076181408, val acc: 0.13165200782864334

Epoch 5
 train loss: 4.413187013029834, train acc: 0.13141427373503364
 val loss: 4.2868934674049495, val acc: 0.15475079957356078

Epoch 6
 train loss: 4.296878065443998, train acc: 0.15313768723444976
 val loss: 4.123306214936506, val acc: 0.17527318763326227

Epoch 7
 train loss: 4.189879021871242, train acc: 0.17162429810700838
 val loss:

In [11]:
test_preds = predict(net, test_loader)
df_submission = pd.read_csv('/kaggle/input/bhw-1-dl-2025-2026/bhw1/sample_submission.csv')
df_submission['Category'] = test_preds
df_submission.to_csv('labels_test.csv', index=False)